# Implementación de Transformers para Procesamiento de Lenguaje Natural (NLP)


### Objetivo
En esta evaluación, implementaremos un modelo basado en arquitecturas de Transformers para una tarea de procesamiento de lenguaje natural (NLP), utilizando el dataset **DailyDialog**. Este conjunto de datos de diálogos permite que el modelo practique en generación de texto y comprensión de contexto en interacciones cotidianas.

Usaremos TensorFlow para construir un modelo transformer básico con las siguientes características:
- **Encoder-Decoder**: para procesar la entrada y generar salida secuencial.
- **Atención Multi-cabezal**: para capturar dependencias a largo plazo en el diálogo.

Al final, evaluaremos el modelo utilizando métricas específicas de NLP, como BLEU o ROUGE.


## 1. Carga y Exploración del Dataset: DailyDialog

In [11]:
# 1) Instalar SentencePiece
!pip install sentencepiece

# 2) Crear un archivo de texto con todos tus turnos (input+target)
#    para que SentencePiece aprenda el vocabulario.
#    Aquí asumimos que `train_inputs` y `train_targets` existen en Python.


In [38]:
import pandas as pd          # Para leer y manipular CSVs
import numpy as np           # Para operaciones numéricas
import torch                 # Framework PyTorch
from torch.utils.data import Dataset, DataLoader
                             # Para definir datasets y loaders en PyTorch

import ast
import re
import sentencepiece as spm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

In [2]:
!wget -q https://raw.githubusercontent.com/JaznaLaProfe/Deep-Learning/main/data/dialog/train.csv
!wget -q https://raw.githubusercontent.com/JaznaLaProfe/Deep-Learning/main/data/dialog/validation.csv
!wget -q https://raw.githubusercontent.com/JaznaLaProfe/Deep-Learning/main/data/dialog/test.csv


In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
validation = pd.read_csv('validation.csv')

In [4]:
train

,dialog,act,emotion
0,"['Say , Jim , how about going for a few beers ...",[3 4 2 2 2 3 4 1 3 4],[0 0 0 0 0 0 4 4 4 4]
1,"['Can you do push-ups ? '\n "" Of course I can ...",[2 1 2 2 1 1],[0 0 6 0 0 0]
2,"['Can you study with the radio on ? '\n ' No ,...",[2 1 2 1 1],[0 0 0 0 0]
3,['Are you all right ? '\n ' I will be all righ...,[2 1 1 1],[0 0 0 0]
4,"['Hey John , nice skates . Are they new ? '\n ...",[2 1 2 1 1 2 1 3 4],[0 0 0 0 0 6 0 6 0]
...,...,...,...
11113,"['Hello , I bought a pen in your shop just bef...",[1 1 1 2 3 2 1 4 1],[0 4 0 0 0 0 0 0 4]
11114,['Do you have any seats available ? ' ' Yes . ...,[2 1 2 1 3 4],[0 0 0 0 0 4]
11115,"['Uncle Ben , how did the Forbidden City get t...",[2 1 2 1 1 1 1 1 2 1 2 1 2 1 3 4],[0 0 6 0 6 0 0 0 0 0 0 0 0 0 4 0]
11116,"['May I help you , sir ? ' ' I want a pair of ...",[2 3 4 3],[0 0 0 0]


In [6]:
def parse_dialog(s: str):
    """
    Convierte el string s (que representa una lista de utterances)
    en una lista de cadenas Python.
    """
    if not isinstance(s, str):
        return []
    # 1. Limpiar saltos de línea y espacios extremos
    s = s.replace('\n', ' ').strip()
    # 2. Quitar corchetes exteriores si existen
    if s.startswith('[') and s.endswith(']'):
        inner = s[1:-1]
    else:
        inner = s
    # 3. Reinsertar comas entre fragmentos de utterance
    #    ej. "'Hola'  '¿Cómo?'" → "'Hola', '¿Cómo?'"
    inner = re.sub(r"'\s+'", "', '", inner)
    inner = re.sub(r'"\s+"', '", "', inner)
    # 4. Construir un literal de lista y evaluar
    literal = f'[{inner}]'
    try:
        parsed = ast.literal_eval(literal)
        # Filtrar sólo cadenas no vacías
        return [u.strip() for u in parsed if isinstance(u, str) and u.strip()]
    except Exception:
        # 5. Fallback: split por dos o más espacios
        parts = re.split(r'\s{2,}', inner)
        return [p.strip(" '\"") for p in parts if p.strip()]

In [7]:
for df in (train, validation, test):
    df['dialog_parsed'] = df['dialog'].apply(parse_dialog)

In [8]:
# Ejemplo en train:
print("Primer diálogo parseado:", train.loc[0, 'dialog_parsed'])
print("Rango de longitudes:",
      train['dialog_parsed'].str.len().min(),
      "a",
      train['dialog_parsed'].str.len().max())


Primer diálogo parseado: ['Say , Jim , how about going for a few beers after dinner ?', 'You know that is tempting but is really not good for our fitness .', "What do you mean ? It will help us to relax .  Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?", "I guess you are right.But what shall we do ? I don't feel like sitting at home .  I suggest a walk over to the gym where we can play singsong and meet some of our friends .  That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .  Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .  Good.Let ' s go now .  All right ."]
Rango de longitudes: 1 a 29


In [9]:
pairs = []
for conv in train['dialog_parsed']:
    for i in range(len(conv)-1):
        pairs.append( (conv[i], conv[i+1]) )


In [20]:
# 1. Inicializa listas vacías
train_inputs  = []
train_targets = []

# 2. Recorre cada diálogo parseado
for conv in train['dialog_parsed']:
    # solo si hay al menos 2 utterances
    if len(conv) < 2:
        continue
    # extrae cada par (i, i+1)
    for i in range(len(conv) - 1):
        train_inputs.append(conv[i])
        train_targets.append(conv[i + 1])

# 3. Muestra el conteo
print("Total de pares de entrenamiento:", len(train_inputs))


Total de pares de entrenamiento: 51341


In [21]:
# -- Extraer pares de validación --
val_inputs, val_targets = [], []
for conv in validation['dialog_parsed']:
    # sólo cuando hay al menos dos utterances
    if len(conv) < 2:
        continue
    for i in range(len(conv) - 1):
        val_inputs.append(conv[i])
        val_targets.append(conv[i+1])

print("Total de pares validación:", len(val_inputs))


Total de pares validación: 4774


In [22]:
# -- Extraer pares de test --
test_inputs, test_targets = [], []
for conv in test['dialog_parsed']:
    if len(conv) < 2:
        continue
    for i in range(len(conv) - 1):
        test_inputs.append(conv[i])
        test_targets.append(conv[i+1])

print("Total de pares test:", len(test_inputs))


Total de pares test: 4544


In [16]:
# 3) Volcar todos los turnos de train en un único fichero
with open('all_texts.txt', 'w', encoding='utf-8') as f:
    for s in train_inputs + train_targets:
        f.write(s.replace('\n',' ') + '\n')

# 4) Entrenar modelo subword con vocab_size=8000 (ajusta si quieres)
spm.SentencePieceTrainer.Train(
    input='all_texts.txt',
    model_prefix='spm_dialog',
    vocab_size=20000,
    model_type='bpe',      # o 'bpe'
    character_coverage=0.9995, # para cubrir casi todos los caracteres
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

# 5) Cargar el tokenizador entrenado
sp = spm.SentencePieceProcessor()
sp.Load('spm_dialog.model')

# 6) Tokenizar un ejemplo
sample = train_inputs[0]
ids     = sp.EncodeAsIds(sample)      # lista de IDs subword
tokens  = sp.EncodeAsPieces(sample)   # lista de subword strings
print(sample)
print(tokens[:10], '→', ids[:10])


Say , Jim , how about going for a few beers after dinner ?
['▁Say', '▁,', '▁Jim', '▁,', '▁how', '▁about', '▁going', '▁for', '▁a', '▁few'] → [3142, 15, 2271, 15, 327, 151, 265, 70, 6, 607]


## 2. Implementación del Modelo Transformer

In [23]:
# 1) Cargar el modelo de tokenización entrenado
sp = spm.SentencePieceProcessor()
sp.Load('spm_dialog.model')

# 2) Parámetros de secuencia y batch
MAX_SEQ_LEN = 40
BATCH_SIZE  = 64

# 3) Función para convertir utterance a lista de IDs de longitud fija
def encode_sequence(text, sp, max_len):
    """
    Tokeniza con sp, añade BOS/EOS, y pad/trunca a max_len.
    """
    ids = sp.EncodeAsIds(text)
    seq = [sp.bos_id()] + ids + [sp.eos_id()]
    if len(seq) > max_len:
        return seq[:max_len]
    # Rellenar con pad_id hasta max_len
    return seq + [sp.pad_id()] * (max_len - len(seq))

# 4) Aplicar a train, validation y test
train_enc = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in train_inputs]
train_dec = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in train_targets]
val_enc   = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in val_inputs]
val_dec   = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in val_targets]
test_enc  = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in test_inputs]
test_dec  = [encode_sequence(s, sp, MAX_SEQ_LEN) for s in test_targets]

# 5) Convertir a tensores
train_enc_tensor = torch.tensor(train_enc, dtype=torch.long)
train_dec_tensor = torch.tensor(train_dec, dtype=torch.long)
val_enc_tensor   = torch.tensor(val_enc,   dtype=torch.long)
val_dec_tensor   = torch.tensor(val_dec,   dtype=torch.long)
test_enc_tensor  = torch.tensor(test_enc,  dtype=torch.long)
test_dec_tensor  = torch.tensor(test_dec,  dtype=torch.long)

# 6) Dataset y DataLoader
train_ds = TensorDataset(train_enc_tensor, train_dec_tensor)
val_ds   = TensorDataset(val_enc_tensor,   val_dec_tensor)
test_ds  = TensorDataset(test_enc_tensor,  test_dec_tensor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

# 7) Verificación rápida de shapes
x_batch, y_batch = next(iter(train_loader))
print("x_batch.shape:", x_batch.shape)  # (batch, MAX_SEQ_LEN)
print("y_batch.shape:", y_batch.shape)

x_batch.shape: torch.Size([64, 40])
y_batch.shape: torch.Size([64, 40])


In [25]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Precomputar sinusoidales
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)  # shape (max_len, 1, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch, d_model)
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

In [26]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8,
                 num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Embeddings para src y tgt
        self.src_tok_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_tok_emb = nn.Embedding(vocab_size, d_model)
        # Positional encodings
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.pos_decoder = PositionalEncoding(d_model, dropout)
        # El Transformer en sí
        self.transformer = nn.Transformer(
            d_model, nhead,
            num_encoder_layers, num_decoder_layers,
            dim_feedforward, dropout
        )
        # Capa final para proyectar a vocab_size
        self.generator = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt,
                src_mask=None, tgt_mask=None,
                src_padding_mask=None, tgt_padding_mask=None,
                memory_key_padding_mask=None):
        # src, tgt: (seq_len, batch)
        src_emb = self.src_tok_emb(src) * math.sqrt(self.d_model)
        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.tgt_tok_emb(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.pos_decoder(tgt_emb)

        output = self.transformer(
            src_emb, tgt_emb,
            src_mask=src_mask, tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        # output: (seq_len, batch, d_model)
        return self.generator(output)  # → (seq_len, batch, vocab_size)


In [27]:
def generate_square_subsequent_mask(sz):
    """Devuelve una máscara (sz, sz) con -inf por encima de la diagonal."""
    mask = torch.triu(torch.ones(sz, sz), diagonal=1)  # triangular superior
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask  # dtype float


In [34]:
# Hiperparámetros
VOCAB_SIZE           = len(sp)          # o len(word2idx)
D_MODEL, NHEAD       = 512, 8
ENC_LAYERS, DEC_LAYERS = 6, 6
DIM_FF               = 2048
DROPOUT              = 0.1

model = Seq2SeqTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=ENC_LAYERS,
    num_decoder_layers=DEC_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=DROPOUT
)

model = model.to(device)



In [48]:
# 1) Mueve las secuencias al device
x_batch = x_batch.to(device)       # (batch, seq_len)
y_batch = y_batch.to(device)

src = x_batch.transpose(0,1)   # (seq_len, batch)
tgt = y_batch.transpose(0,1)

seq_len  = src.size(0)
tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)


In [49]:
src_padding_mask   = (x_batch == pad_idx)    # ya en device
tgt_padding_mask   = (y_batch == pad_idx)
memory_key_padding_mask = src_padding_mask

In [62]:
pad_idx = sp.pad_id()
output = model(
    src, tgt,
    src_mask=None,
    tgt_mask=tgt_mask,
    src_padding_mask=(x_batch == pad_idx),
    tgt_padding_mask=(y_batch == pad_idx),
    memory_key_padding_mask=(x_batch == pad_idx)
)
# output: (seq_len, batch, vocab_size)


In [64]:
output = model(
    src, tgt,
    src_mask=None,
    tgt_mask=tgt_mask,
    src_padding_mask=src_padding_mask,        # ahora coincide con tu forward
    tgt_padding_mask=tgt_padding_mask,
    memory_key_padding_mask=memory_mask
)


In [65]:
import torch.nn as nn

# Asumiendo que pad_idx ya está definido:
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)


In [66]:
# Aplanar primero
logits = output.transpose(0,1).reshape(-1, VOCAB_SIZE)  # (batch*seq_len, vocab)
targets = y_batch.reshape(-1)                           # (batch*seq_len)
loss = criterion(logits, targets)

In [67]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



## 3. Entrenamiento del Modelo

In [70]:
import time
import torch.nn as nn

NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ——— Entrenamiento ———
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        # 1) Mover a device
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        # 2) Preparar src, tgt
        src = x_batch.transpose(0,1)   # (seq_len, batch)
        tgt = y_batch.transpose(0,1)
        # 3) Máscaras
        tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
        src_padding_mask = (x_batch == pad_idx)
        tgt_padding_mask = (y_batch == pad_idx)
        memory_mask = src_padding_mask
        # 4) Forward + loss
        optimizer.zero_grad()
        output = model(
          src, tgt,
          src_mask=None,
          tgt_mask=tgt_mask,
          src_padding_mask=src_padding_mask,        # ahora coincide con tu forward
          tgt_padding_mask=tgt_padding_mask,
          memory_key_padding_mask=memory_mask
          )
        # 5) Aplanar y calcular loss
        logits  = output.transpose(0,1).reshape(-1, VOCAB_SIZE)
        targets = y_batch.reshape(-1)
        loss    = criterion(logits, targets)
        # 6) Backward + step
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ——— Validación ———
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            src = x_batch.transpose(0,1)
            tgt = y_batch.transpose(0,1)
            tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
            src_padding_mask = (x_batch == pad_idx)
            tgt_padding_mask = (y_batch == pad_idx)
            memory_mask = src_padding_mask

            output = model(
              src, tgt,
              src_mask=None,
              tgt_mask=tgt_mask,
              src_padding_mask=src_padding_mask,        # ahora coincide con tu forward
              tgt_padding_mask=tgt_padding_mask,
              memory_key_padding_mask=memory_mask
            )
            logits  = output.transpose(0,1).reshape(-1, VOCAB_SIZE)
            targets = y_batch.reshape(-1)
            val_loss += criterion(logits, targets).item()

    val_loss /= len(val_loader)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch:>2} | time: {elapsed:5.2f}s | "
          f"train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")


Epoch  1 | time: 235.05s | train_loss: 0.5112 | val_loss: 0.2878
Epoch  2 | time: 233.98s | train_loss: 0.2682 | val_loss: 0.1618
Epoch  3 | time: 234.05s | train_loss: 0.1541 | val_loss: 0.0993
Epoch  4 | time: 234.04s | train_loss: 0.0902 | val_loss: 0.0626
Epoch  5 | time: 233.88s | train_loss: 0.0518 | val_loss: 0.0420
Epoch  6 | time: 233.91s | train_loss: 0.0283 | val_loss: 0.0313
Epoch  7 | time: 233.87s | train_loss: 0.0153 | val_loss: 0.0242
Epoch  8 | time: 233.93s | train_loss: 0.0081 | val_loss: 0.0213
Epoch  9 | time: 233.80s | train_loss: 0.0036 | val_loss: 0.0208
Epoch 10 | time: 233.97s | train_loss: 0.0015 | val_loss: 0.0206


## 4. Evaluación del Modelo

In [ ]:
# Ejemplo de evaluación del modelo usando BLEU o ROUGE
# predictions = model.predict(test_data)
# print("BLEU Score:", sentence_bleu(reference_sentences, predictions))


## 5. Ajuste de Hiperparámetros

In [ ]:

# Probar diferentes configuraciones de hiperparámetros
# Ejemplo: modificar num_heads, ff_dim, número de capas

# Documentar los resultados y evaluar cada configuración
# for num_heads in [2, 4, 8]:
#     for ff_dim in [32, 64, 128]:
#         # Redefinir y entrenar modelo con nuevos hiperparámetros
#         # Registrar métricas y comparar rendimiento


## 6. Presentación de Resultados y Conclusiones


En esta sección, resumiremos los resultados obtenidos, mostrando cómo los ajustes de los hiperparámetros impactaron en el rendimiento del modelo.
- **Resultados Finales**: Comparación de BLEU, ROUGE, y otras métricas para cada configuración.
- **Conclusiones**: Reflexión sobre el proceso, dificultades encontradas y aprendizajes obtenidos.

¡Gracias por revisar nuestro proyecto! Esperamos que esta implementación demuestre nuestro dominio en el uso de transformers para NLP.
